## Problem

A company ships from plants → distribution centers (DCs) → customers. Plants can't ship directly to customers — everything routes through a DC. Minimize total shipping cost.

In [1]:
import pandas as pd
import numpy as np
import pulp as pl

In [2]:
# Define the data
Plants = ['P1', 'P2']
supply_capacity = {'P1': 150, 'P2': 200}
distribution_centers = ['D1', 'D2', 'D3']
dc_throughput_capacity = {'D1': 150, 'D2': 130, 'D3': 120}
customers = ['C1', 'C2', 'C3', 'C4']
customer_demand = {'C1': 50, 'C2': 80, 'C3': 70, 'C4': 90}
shipping_cost_p2c = {('P1', 'D1'): 4, ('P1', 'D2'): 6, ('P1', 'D3'): 9,
                 ('P2', 'D1'): 5, ('P2', 'D2'): 4, ('P2', 'D3'): 7}
shipping_cost_d2c = {('D1', 'C1'): 3, ('D1', 'C2'): 5, ('D1', 'C3'): 8, ('D1', 'C4'): 4,
                 ('D2', 'C1'): 6, ('D2', 'C2'): 4, ('D2', 'C3'): 3, ('D2', 'C4'): 5,
                 ('D3', 'C1'): 7, ('D3', 'C2'): 3, ('D3', 'C3'): 2, ('D3', 'C4'): 4}


In [ ]:
class supplychain_optimization_model:
    def __init__(self, plants, supply_capacity, distribution_centers, dc_throughput_capacity, customers, customer_demand, shipping_cost_p2c, shipping_cost_d2c):
        self.plants = plants
        self.supply_capacity = supply_capacity
        self.distribution_centers = distribution_centers
        self.dc_throughput_capacity = dc_throughput_capacity
        self.customers = customers
        self.customer_demand = customer_demand
        self.shipping_cost_p2c = shipping_cost_p2c
        self.shipping_cost_d2c = shipping_cost_d2c
        self.model = pl.LpProblem("Capacitated_Facility_Location", pl.LpMinimize)
        self.create_decision_variables()
        self.add_constraints()
        self.set_objective()
    def create_decision_variables(self):
        self.x_p2d = pl.LpVariable.dicts("x_p2d", (self.plants, self.distribution_centers), lowBound=0, cat='Continuous')
        self.x_d2c = pl.LpVariable.dicts("x_d2c", (self.distribution_centers, self.customers), lowBound=0, cat='Continuous')
    def add_constraints(self):
        # Supply capacity constraints
        for p in self.plants:
            self.model += pl.lpSum(self.x_p2d[p][d] for d in self.distribution_centers) <= self.supply_capacity[p], f"Supply_Capacity_{p}"
        # Distribution center throughput constraints
        for d in self.distribution_centers:
            self.model += pl.lpSum(self.x_p2d[p][d] for p in self.plants) == pl.lpSum(self.x_d2c[d][c] for c in self.customers), f"Throughput_{d}"
            self.model += pl.lpSum(self.x_d2c[d][c] for c in self.customers) <= self.dc_throughput_capacity[d], f"Throughput_Capacity_{d}"
        # Customer demand constraints
        for c in self.customers:
            self.model += pl.lpSum(self.x_d2c[d][c] for d in self.distribution_centers) >= self.customer_demand[c], f"Customer_Demand_{c}"
    def set_objective(self):
        self.model += pl.lpSum(self.shipping_cost_p2c[p, d] * self.x_p2d[p][d] for p in self.plants for d in self.distribution_centers) + \
                      pl.lpSum(self.shipping_cost_d2c[d, c] * self.x_d2c[d][c] for d in self.distribution_centers for c in self.customers), "Total_Cost"
    def solve(self, verbose=False):
        if verbose:
            self.model.solve()
        else:
            self.model.solve(pl.PULP_CBC_CMD(msg=0))
        return pl.LpStatus[self.model.status], pl.value(self.model.objective), self.get_solution()
    def get_solution(self):
        solution = {
            'x_p2d': {(p, d): self.x_p2d[p][d].varValue for p in self.plants for d in self.distribution_centers},
            'x_d2c': {(d, c): self.x_d2c[d][c].varValue for d in self.distribution_centers for c in self.customers}
        }
        return solution

if __name__ == "__main__":
    model = supplychain_optimization_model(Plants, supply_capacity, distribution_centers, dc_throughput_capacity, customers, customer_demand, shipping_cost_p2c, shipping_cost_d2c)
    status, total_cost, solution = model.solve(verbose=True)
    print(f"Status: {status}")
    print(f"Total Cost: {total_cost}")
    print("Solution:")
    for key in solution:
        print(f"{key}:")
        for k, v in solution[key].items():
            print(f"  {k}: {v}")

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /usr/local/python/3.12.1/lib/python3.12/site-packages/pulp/apis/../solverdir/cbc/linux/i64/cbc /tmp/8e2567b5340648bfba2f9da983f001e8-pulp.mps -timeMode elapsed -solve -printingOptions all -solution /tmp/8e2567b5340648bfba2f9da983f001e8-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 17 COLUMNS
At line 84 RHS
At line 97 BOUNDS
At line 98 ENDATA
Problem MODEL has 12 rows, 18 columns and 48 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Presolve 12 (0) rows, 18 (0) columns and 48 (0) elements
0  Obj 0 Primal inf 290 (4)
11  Obj 2230
Optimal - objective value 2230
Optimal objective 2230 - 11 iterations time 0.002
Option for printingOptions changed from normal to all
Total time (CPU seconds):       0.00   (Wallclock seconds):       0.00

Status: Optimal
Total Cost: 2230.0
Solution:
x_p2d:
  ('P1', 'D1'): 150.0
  ('P1', 'D2'